# Parametric PINN with original `pinn_ndof_chain_tf2` settings

This notebook keeps the original segment solver idea (Adam + L-BFGS) to generate 50-impact trajectories for sampled `(phi1, phi2)`, then trains one parametric network for unseen settings (output: x only).

In [ ]:
from pathlib import Path
import importlib.util
import sys
import numpy as np
import matplotlib.pyplot as plt

module_path = Path('parametric PINN/parametric_pinn_50_impacts.py')
spec = importlib.util.spec_from_file_location('parametric_pinn_50_impacts', module_path)
mod = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = mod
spec.loader.exec_module(mod)

LegacySimConfig = mod.LegacySimConfig
ParametricDataConfig = mod.ParametricDataConfig
ParametricModelConfig = mod.ParametricModelConfig
LegacyFullHorizonGenerator = mod.LegacyFullHorizonGenerator
ParametricFullHorizonPINN = mod.ParametricFullHorizonPINN
build_parametric_dataset = mod.build_parametric_dataset


In [ ]:
# Keep original physical/training settings from pinn_ndof_chain_sim_tf2.ipynb
sim_cfg = LegacySimConfig(
    n_dof=20, m_x=1.0, m_y=0.3, k=1.0, c=0.0, D=1.0, r=1.0,
    num_neurons=64, hyp_ini_weight_loss=(1.0, 1.0), optimizer_LB_value=True,
    n_segments=50, T_seg=1.0, num_seg=1000, nIter_seg=1000,
)

# Parametric sampling in requested range
data_cfg = ParametricDataConfig(
    phi1_range=(1.0, 2.0),
    phi2_range=(10.0, 20.0),
    n_param_samples=20,
)

# One parametric network with Adam + L-BFGS
model_cfg = ParametricModelConfig(
    layers=(3, 128, 128, 128, sim_cfg.n_dof),
    adam_epochs=2000,
    optimizer_LB=True,
)


In [ ]:
# Step 1: generate full 50-impact trajectories using original segment PINN
generator = LegacyFullHorizonGenerator(sim_cfg)
dataset = build_parametric_dataset(generator, data_cfg)
print('dataset x size:', dataset['x'].shape)


In [ ]:
# Step 2: train ONE parametric network over all sampled parameter settings
model = ParametricFullHorizonPINN(sim_cfg, model_cfg)
model.train(dataset, data_cfg.phi1_range, data_cfg.phi2_range)


In [ ]:
# Unseen in-range parameter query
phi1_test, phi2_test = 1.37, 16.25
t_eval = np.linspace(0.0, np.max(dataset['t_local']), 20000)
x_pred = model.predict_x(t_eval, phi1_test, phi2_test)
impact_times = model.extract_impact_times(t_eval, phi1_test, phi2_test, D=sim_cfg.D, max_impacts=50)

print('x_pred shape:', x_pred.shape)
print('Detected impacts:', impact_times.shape[0])
print('First 10 impact times:', impact_times[:10])


In [ ]:
# Plot last DOF response and detected impacts
x_last = x_pred[:, -1]
plt.figure(figsize=(12, 4))
plt.plot(t_eval, x_last, lw=1.0, label='x_last(t) prediction')
if impact_times.size > 0:
    y_imp = np.interp(impact_times, t_eval, x_last)
    plt.scatter(impact_times, y_imp, s=15, c='r', label='detected impacts')
plt.xlabel('t')
plt.ylabel('x_last')
plt.title(f'Unseen parameter prediction: phi1={phi1_test}, phi2={phi2_test}')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()
